# STEP 1: Install Dependencies (with AgentOps)

In [1]:
!pip install -q crewai crewai-tools agentops google-generativeai duckduckgo_search yfinance pandas beautifulsoup4 python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.2/320.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 602.7/602.7 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 17.0 

# STEP 2: Imports (add AgentOps)

In [1]:
import os
from dotenv import load_dotenv
from getpass import getpass
import yfinance as yf
from duckduckgo_search import DDGS
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
import agentops

# Load environment variables
load_dotenv()


False

# STEP 3: Load Keys (Gemini + AgentOps)

In [2]:
# Load Gemini API Key
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

# Load AgentOps API Key
os.environ["AGENTOPS_API_KEY"] = getpass("Enter your AgentOps API Key: ")

# Initialize AgentOps
agentops.init(api_key=os.environ["AGENTOPS_API_KEY"])


Enter your Gemini API Key: ··········
Enter your AgentOps API Key: ··········


🖇 AgentOps: Session Replay for default trace: https://app.agentops.ai/sessions?trace_id=803105562c59332a540b3def1b13f685


#  STEP 4: LLM Setup (Gemini via CrewAI)

In [3]:
llm = LLM(
    model="gemini/gemini-2.0-flash",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.7
)


# STEP 5: Define Tools

In [4]:
class StockSearchTool(BaseTool):
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=2)
            return "\n".join([r['body'] for r in results])

class YahooFinanceTool(BaseTool):
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="1mo")
        return hist.tail(3).to_string()


# STEP 6: Define Agents (Monitored by AgentOps)

In [5]:
stock_analyst = Agent(
    role='Stock Analyst',
    goal='Analyze recent stock data and news',
    backstory='Expert in financial trends, macro indicators, and company performance',
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True  # ✅ Enables AgentOps logging
)

report_writer = Agent(
    role='Report Generator',
    goal='Write investor-friendly summaries of stock analysis',
    backstory='Professional writer with expertise in finance reporting',
    verbose=True,
    allow_delegation=False,
    llm=llm,
    agentops_enabled=True  # ✅ Enables AgentOps logging
)


# STEP 7: Define Tasks

In [6]:
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,
    tools=[search_tool]
)

analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,
    tools=[finance_tool]
)

report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer
)


# STEP 8: Assemble the Crew

In [7]:
crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task, report_task],
    verbose=False  # Keep output clean
)


# STEP 9: Run the Agent Crew with AgentOps Logging

In [8]:
result = crew.kickoff()

print("\n📊 Final Stock Analysis Report:\n")
print(result)


# Agent: Stock Analyst
## Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.


# Agent: Stock Analyst
## Thought: I need to find the latest news and updates about Apple (AAPL) stock using the StockNewsSearcher tool.
## Using tool: StockNewsSearcher
## Tool Input: 
"{\"query\": \"AAPL stock\"}"
## Tool Output: 
Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with your stock trading and investing.
Get the latest Apple Inc (AAPL) real-time quote, historical performance, charts, and other financial information to help you make more informed trading and investment decisions.


# Agent: Stock Analyst
## Final Answer: 
Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with your stock trading and investing.
Get the latest Apple Inc (AAPL) real-time quote, historical performance, charts, and other financial information to help you make more informed trading and inve

🖇 AgentOps: Session Replay for default.session trace: https://app.agentops.ai/sessions?trace_id=803105562c59332a540b3def1b13f685



📊 Final Stock Analysis Report:

**Apple Inc. (AAPL) – Investor Report (as of June 21, 2025)**

**Market Summary:**

Apple Inc. (AAPL) stock has demonstrated an overall uptrend throughout the past month, characterized by price fluctuations and increasing closing prices, which suggests positive investor sentiment. Trading volumes have shown notable spikes on specific days, potentially correlating with significant news or market events. The stock has recently consolidated within the $214-$217 range.

**Key Technical Highlights:**

*   **Uptrend:** Evident throughout the month of May and June.
*   **Volatility:** Present, with fluctuations in the stock price.
*   **Resistance:** Observed around $217.54 (June 21st).
*   **Support:** Found near $207.58 (June 14th).
*   **Volume Spikes:** Notable increases in trading volume on May 23rd and June 20th.

**Recent Performance:**

| Date          | Open       | High       | Low        | Close      | Volume     |
|---------------|------------|----